# LeetCode 290 — Word Pattern bằng Python

**Link đề:** https://leetcode.com/problems/word-pattern/
**Độ khó:** Easy · **Chủ đề:** Hash Table, String

```python
class Solution:
    def wordPattern(self, pattern: str, s: str) -> bool:
        ...
```

## Mục tiêu học
1. Tách câu thành từ bằng `str.split()`, ghép cặp bằng `zip`.
2. Hiểu **song ánh** (bijection): mỗi chữ ↔ đúng một từ, **hai chiều**.
3. Viết baseline "so sánh vị trí xuất hiện đầu tiên", rồi lời giải chính `O(n)` với **hai `dict`**.
4. Biết các cách gọn: một `dict` + một `set`, mẹo đếm `len(set(zip(...)))`.
5. Tránh các bẫy: độ dài khác nhau, chỉ kiểm một chiều, `zip` tự cắt ngắn.

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Cell TODO chỉ định nghĩa hàm, không tự chạy.

## 1. Đề bài

Cho `pattern` (chuỗi chữ cái) và câu `s` (các từ cách nhau một dấu cách). Hỏi `s` có **theo đúng** mẫu `pattern` không.

"Theo đúng" = có một **song ánh** giữa chữ cái trong `pattern` và **từ** trong `s`:
- Mỗi chữ cái ánh xạ tới **đúng một** từ.
- Mỗi từ ánh xạ tới **đúng một** chữ cái.
- Không có hai chữ cái cùng ánh xạ tới một từ, và ngược lại.

| pattern | s | Kết quả | Vì sao |
|---|---|:---:|---|
| `"abba"` | `"dog cat cat dog"` | `True` | a↔dog, b↔cat |
| `"abba"` | `"dog cat cat fish"` | `False` | a phải là dog, nhưng cuối lại là fish |
| `"aaaa"` | `"dog cat cat dog"` | `False` | a không thể vừa là dog vừa là cat |
| `"abba"` | `"dog dog dog dog"` | `False` | a và b cùng là dog (vi phạm chiều ngược) |

**Constraints:** `1 <= len(pattern) <= 300`, chữ thường; `1 <= len(s) <= 3000`, chữ thường và dấu cách, không có dấu cách đầu/cuối, các từ cách nhau **đúng một** dấu cách.

In [1]:
pattern, s = "abba", "dog cat cat dog"
words = s.split()
for ch, w in zip(pattern, words):
    print(ch, "->", w)

a -> dog
b -> cat
b -> cat
a -> dog


## 2. Nền tảng Python

### 2.1. `str.split()` — tách câu thành list từ

- `s.split()` (không tham số) tách theo **mọi** khoảng trắng, bỏ chuỗi rỗng.
- `s.split(" ")` tách theo đúng một dấu cách; nếu có hai dấu cách liền nhau sẽ sinh chuỗi rỗng `""`.

Với đề này (đúng một dấu cách) hai cách cho kết quả giống nhau; `split()` an toàn hơn.

In [2]:
print("dog cat cat dog".split())
print("dog  cat".split(), "vs", "dog  cat".split(" "))
print(len("dog cat cat dog".split()))

['dog', 'cat', 'cat', 'dog']
['dog', 'cat'] vs ['dog', '', 'cat']
4


### 2.2. `zip` — ghép từng cặp, **cắt theo dãy ngắn hơn**

`zip(a, b)` cho các cặp `(a[0], b[0]), (a[1], b[1]), ...` và **dừng khi một dãy hết**. Đây là nguồn gốc của một lỗi kinh điển: nếu quên kiểm độ dài, phần thừa bị bỏ qua âm thầm.

In [3]:
print(list(zip("abba", ["dog", "cat", "cat", "dog"])))
print(list(zip("abc", ["x", "y"])), "<- 'c' bị bỏ mất!")

[('a', 'dog'), ('b', 'cat'), ('b', 'cat'), ('a', 'dog')]
[('a', 'x'), ('b', 'y')] <- 'c' bị bỏ mất!


### 2.3. `dict` để lưu ánh xạ, `in` để kiểm tra

- `m[k] = v` gán; `k in m` kiểm tra key; `m[k]` đọc giá trị.
- Ánh xạ **một chiều** `chữ → từ` chỉ đảm bảo "mỗi chữ một từ". Cần thêm chiều `từ → chữ` để đảm bảo "mỗi từ một chữ".

In [4]:
c2w = {}
c2w["a"] = "dog"
c2w["b"] = "cat"
print(c2w, "| 'a' in c2w:", "a" in c2w, "| c2w['a'] =", c2w["a"])

{'a': 'dog', 'b': 'cat'} | 'a' in c2w: True | c2w['a'] = dog


### 2.4. Song ánh là gì — ví dụ đời thường

Mỗi sinh viên có **một** mã số, mỗi mã số thuộc **một** sinh viên. Nếu hai sinh viên dùng chung mã → hỏng chiều "mã → sinh viên"; nếu một sinh viên có hai mã → hỏng chiều "sinh viên → mã".

Bảng minh họa với `pattern = "abba"`:

| s | chữ → từ ổn? | từ → chữ ổn? | Song ánh? |
|---|:---:|:---:|:---:|
| dog cat cat dog | ✔ | ✔ | ✔ |
| dog cat cat fish | ✘ (a → dog, fish) | ✔ | ✘ |
| dog dog dog dog | ✔ (a → dog, b → dog) | ✘ (dog → a, b) | ✘ |

## 3. Cách 1 — Baseline: so sánh "vị trí xuất hiện đầu tiên"

Ý tưởng: hai dãy có **cùng hình dạng** khi và chỉ khi, tại mỗi vị trí `i`, phần tử thứ `i` xuất hiện lần đầu ở **cùng chỉ số** trong cả hai dãy.

```text
pattern: a b b a   -> index lần đầu: 0 1 1 0
words  : dog cat cat dog -> 0 1 1 0      giống nhau -> True
words  : dog dog dog dog -> 0 0 0 0      khác       -> False
```

`list.index(x)` trả về chỉ số xuất hiện **đầu tiên** của `x` — tốn `O(n)` mỗi lần → tổng `O(n²)`. Với giới hạn đề nhỏ (≤ 300 từ) vẫn chạy được, và cách này **tự động** kiểm cả hai chiều.

In [5]:
def word_pattern_index(pattern: str, s: str) -> bool:
    words = s.split()
    if len(pattern) != len(words):
        return False
    return [pattern.index(c) for c in pattern] == [words.index(w) for w in words]

print([pattern.index(c) for c in "abba"], ["dog", "cat", "cat", "dog"].index("cat"))
for p, s in [("abba", "dog cat cat dog"), ("abba", "dog cat cat fish"), ("aaaa", "dog cat cat dog"), ("abba", "dog dog dog dog")]:
    print(f"{p!r}, {s!r} -> {word_pattern_index(p, s)}")

[0, 1, 1, 0] 1
'abba', 'dog cat cat dog' -> True
'abba', 'dog cat cat fish' -> False
'aaaa', 'dog cat cat dog' -> False
'abba', 'dog dog dog dog' -> False


## 4. Nút thắt và insight — kiểm **hai chiều** trong một lượt

Cách 1 tốn `O(n²)` vì `index` quét lại từ đầu. Ta muốn **nhớ** những gì đã thấy để tra `O(1)`: dùng `dict`.

Insight: song ánh = hai ánh xạ một chiều **cùng nhất quán**:
- `c2w`: chữ → từ. Gặp lại chữ `c` thì từ phải đúng là `c2w[c]`.
- `w2c`: từ → chữ. Gặp lại từ `w` thì chữ phải đúng là `w2c[w]`.

Chỉ một chiều là **không đủ** — ví dụ `"abba"` với `"dog dog dog dog"` qua được `c2w` nhưng hỏng ở `w2c`.

In [6]:
c2w = {}
ok_one_way = True
for c, w in zip("abba", "dog dog dog dog".split()):
    if c in c2w and c2w[c] != w:
        ok_one_way = False
    c2w[c] = w
print("chỉ kiểm chữ -> từ:", ok_one_way, c2w, "<- tưởng đúng nhưng a, b cùng là 'dog'")

chỉ kiểm chữ -> từ: True {'a': 'dog', 'b': 'dog'} <- tưởng đúng nhưng a, b cùng là 'dog'


## 5. Lời giải chính — hai `dict`, giảng từng đoạn

**Đoạn 1 — tách và kiểm độ dài:**
```python
words = s.split()
if len(pattern) != len(words):
    return False
```

**Đoạn 2 — duyệt từng cặp:** `for c, w in zip(pattern, words):`

**Đoạn 3 — kiểm chiều chữ → từ:**
```python
if c in c2w and c2w[c] != w:
    return False
```

**Đoạn 4 — kiểm chiều từ → chữ:**
```python
if w in w2c and w2c[w] != c:
    return False
```

**Đoạn 5 — ghi nhận cặp:** `c2w[c] = w; w2c[w] = c`.

**Đoạn 6:** duyệt hết không mâu thuẫn → `return True`.

In [7]:
def word_pattern_two_dicts(pattern: str, s: str) -> bool:
    words = s.split()
    if len(pattern) != len(words):
        return False
    c2w, w2c = {}, {}
    for c, w in zip(pattern, words):
        if c in c2w and c2w[c] != w:
            return False
        if w in w2c and w2c[w] != c:
            return False
        c2w[c] = w
        w2c[w] = c
    return True

for p, s in [("abba", "dog cat cat dog"), ("abba", "dog cat cat fish"), ("aaaa", "dog cat cat dog"), ("abba", "dog dog dog dog")]:
    print(f"{p!r}, {s!r} -> {word_pattern_two_dicts(p, s)}")

'abba', 'dog cat cat dog' -> True
'abba', 'dog cat cat fish' -> False
'aaaa', 'dog cat cat dog' -> False
'abba', 'dog dog dog dog' -> False


## 6. TRACE từng bước

### 6.1. `pattern = "abba"`, `s = "dog cat cat dog"`

| i | c | w | c2w[c] trước | w2c[w] trước | quyết định | c2w sau | w2c sau |
|---:|:---:|---|---|---|---|---|---|
| 0 | a | dog | — | — | ghi | {a:dog} | {dog:a} |
| 1 | b | cat | — | — | ghi | {a:dog, b:cat} | {dog:a, cat:b} |
| 2 | b | cat | cat ✔ | b ✔ | ghi (không đổi) | như trên | như trên |
| 3 | a | dog | dog ✔ | a ✔ | ghi (không đổi) | như trên | như trên |

→ `True`.

### 6.2. `pattern = "abba"`, `s = "dog dog dog dog"`

| i | c | w | c2w[c] trước | w2c[w] trước | quyết định |
|---:|:---:|---|---|---|---|
| 0 | a | dog | — | — | ghi |
| 1 | b | dog | — | **a ≠ b** | `False` |

In [8]:
def trace_word_pattern(pattern: str, s: str) -> bool:
    words = s.split()
    print(f"pattern = {pattern!r}, words = {words}")
    if len(pattern) != len(words):
        print(f"  độ dài khác nhau ({len(pattern)} vs {len(words)}) -> False")
        return False
    c2w, w2c = {}, {}
    for i, (c, w) in enumerate(zip(pattern, words)):
        prev_w, prev_c = c2w.get(c, "—"), w2c.get(w, "—")
        print(f"  i={i} c={c!r} w={w!r:<7} | c2w[c] trước = {prev_w!r:<7} | w2c[w] trước = {prev_c!r:<5}", end=" | ")
        if c in c2w and c2w[c] != w:
            print(f"mâu thuẫn: {c!r} đã là {c2w[c]!r} -> False")
            return False
        if w in w2c and w2c[w] != c:
            print(f"mâu thuẫn: {w!r} đã thuộc {w2c[w]!r} -> False")
            return False
        c2w[c] = w
        w2c[w] = c
        print(f"OK, c2w = {c2w}")
    print("  -> True")
    return True

trace_word_pattern("abba", "dog cat cat dog")
print()
trace_word_pattern("abba", "dog dog dog dog")
print()
trace_word_pattern("abba", "dog cat cat fish")
print()
trace_word_pattern("aaa", "aa aa aa aa")

pattern = 'abba', words = ['dog', 'cat', 'cat', 'dog']
  i=0 c='a' w='dog'   | c2w[c] trước = '—'     | w2c[w] trước = '—'   | OK, c2w = {'a': 'dog'}
  i=1 c='b' w='cat'   | c2w[c] trước = '—'     | w2c[w] trước = '—'   | OK, c2w = {'a': 'dog', 'b': 'cat'}
  i=2 c='b' w='cat'   | c2w[c] trước = 'cat'   | w2c[w] trước = 'b'   | OK, c2w = {'a': 'dog', 'b': 'cat'}
  i=3 c='a' w='dog'   | c2w[c] trước = 'dog'   | w2c[w] trước = 'a'   | OK, c2w = {'a': 'dog', 'b': 'cat'}
  -> True

pattern = 'abba', words = ['dog', 'dog', 'dog', 'dog']
  i=0 c='a' w='dog'   | c2w[c] trước = '—'     | w2c[w] trước = '—'   | OK, c2w = {'a': 'dog'}
  i=1 c='b' w='dog'   | c2w[c] trước = '—'     | w2c[w] trước = 'a'   | mâu thuẫn: 'dog' đã thuộc 'a' -> False

pattern = 'abba', words = ['dog', 'cat', 'cat', 'fish']
  i=0 c='a' w='dog'   | c2w[c] trước = '—'     | w2c[w] trước = '—'   | OK, c2w = {'a': 'dog'}
  i=1 c='b' w='cat'   | c2w[c] trước = '—'     | w2c[w] trước = '—'   | OK, c2w = {'a': 'dog', 'b': 'cat'

False

`for i, (c, w) in enumerate(zip(...))`: `enumerate` cho chỉ số `i` và phần tử là **cặp** `(c, w)`; dấu ngoặc tròn "mở" cặp thành hai biến.

## 7. Cách 3 — Một `dict` + một `set` "từ đã được dùng"

Chiều `từ → chữ` chỉ cần biết "từ này **đã có chủ** chưa". Khi gặp chữ `c` **lần đầu**, nếu `w` đã được chữ khác dùng → `False`.

In [9]:
def word_pattern_dict_set(pattern: str, s: str) -> bool:
    words = s.split()
    if len(pattern) != len(words):
        return False
    c2w = {}
    used = set()
    for c, w in zip(pattern, words):
        if c in c2w:
            if c2w[c] != w:
                return False
        else:
            if w in used:              # w đã thuộc chữ khác
                return False
            c2w[c] = w
            used.add(w)
    return True

for p, s in [("abba", "dog cat cat dog"), ("abba", "dog cat cat fish"), ("aaaa", "dog cat cat dog"), ("abba", "dog dog dog dog")]:
    print(f"{p!r}, {s!r} -> {word_pattern_dict_set(p, s)}")

'abba', 'dog cat cat dog' -> True
'abba', 'dog cat cat fish' -> False
'aaaa', 'dog cat cat dog' -> False
'abba', 'dog dog dog dog' -> False


**Vì sao đủ?** Khi `c` đã có trong `c2w`, ta kiểm `c2w[c] == w`; nếu đúng thì `w` cũng chính là từ của `c`, chiều ngược tự nhất quán. Khi `c` mới, `w` phải **chưa có chủ**.

## 8. Cách 4 — Mẹo đếm phần tử phân biệt với `set(zip(...))`

Nếu có song ánh thì số chữ phân biệt = số từ phân biệt = số **cặp** (chữ, từ) phân biệt. Ngược lại, bất kỳ vi phạm nào cũng làm số cặp lớn hơn một trong hai số kia.

```python
len(set(pattern)) == len(set(words)) == len(set(zip(pattern, words)))
```

(cộng điều kiện cùng độ dài, vì `zip` cắt ngắn).

In [10]:
def word_pattern_sets(pattern: str, s: str) -> bool:
    words = s.split()
    if len(pattern) != len(words):
        return False
    return len(set(pattern)) == len(set(words)) == len(set(zip(pattern, words)))

for p, s in [("abba", "dog cat cat dog"), ("abba", "dog cat cat fish"), ("aaaa", "dog cat cat dog"), ("abba", "dog dog dog dog")]:
    words = s.split()
    print(f"{p!r}, {s!r}: |chữ| = {len(set(p))}, |từ| = {len(set(words))}, |cặp| = {len(set(zip(p, words)))} -> {word_pattern_sets(p, s)}")

'abba', 'dog cat cat dog': |chữ| = 2, |từ| = 2, |cặp| = 2 -> True
'abba', 'dog cat cat fish': |chữ| = 2, |từ| = 3, |cặp| = 3 -> False
'aaaa', 'dog cat cat dog': |chữ| = 1, |từ| = 2, |cặp| = 2 -> False
'abba', 'dog dog dog dog': |chữ| = 2, |từ| = 1, |cặp| = 2 -> False


**Chứng minh ngắn:** Gọi `P` = tập chữ, `W` = tập từ, `Z` = tập cặp. Luôn có `|Z| >= |P|` và `|Z| >= |W|` (mỗi chữ/từ xuất hiện trong ít nhất một cặp). `|Z| == |P|` ⇔ mỗi chữ chỉ đi với **một** từ; `|Z| == |W|` ⇔ mỗi từ chỉ đi với **một** chữ. Cả hai ⇔ song ánh.

## 9. Code nộp LeetCode

In [11]:
class Solution:
    def wordPattern(self, pattern: str, s: str) -> bool:
        words = s.split()
        if len(pattern) != len(words):
            return False
        c2w, w2c = {}, {}
        for c, w in zip(pattern, words):
            if c2w.get(c, w) != w or w2c.get(w, c) != c:
                return False
            c2w[c] = w
            w2c[w] = c
        return True

sol = Solution()
print([sol.wordPattern(p, s) for p, s in [("abba", "dog cat cat dog"), ("abba", "dog cat cat fish"), ("aaaa", "dog cat cat dog")]])

[True, False, False]


Mẹo `c2w.get(c, w) != w`: nếu `c` **chưa có** thì `get` trả mặc định `w` → so sánh `w != w` là `False` (không mâu thuẫn); nếu có thì so với từ đã lưu. Gọn hai `if` thành một dòng.

## 10. Vì sao đúng? (invariant) + độ phức tạp

**Invariant:** sau khi xử lý `i` cặp đầu, `c2w` và `w2c` là hai ánh xạ **ngược nhau** (`c2w[c] = w` ⇔ `w2c[w] = c`) và nhất quán với mọi cặp đã xét. Nếu cặp `(c, w)` mới mâu thuẫn ở chiều nào, không thể có song ánh nào chứa cả các cặp cũ lẫn cặp mới → `False` đúng. Nếu qua hết mọi cặp (và độ dài bằng nhau), `c2w` chính là song ánh cần tìm → `True` đúng.

Gọi `n = len(pattern)`, `m = len(s)` (số ký tự của câu).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. So index lần đầu | `O(n²·L)` (`index` quét + so chuỗi dài `L`) | `O(n)` | ngắn, tự kiểm 2 chiều |
| 2. Hai `dict` | `O(m)` (split + hash từ) | `O(m)` | **lời giải chính** |
| 3. `dict` + `set` | `O(m)` | `O(m)` | một chiều + "đã có chủ" |
| 4. Đếm `set(zip)` | `O(m)` | `O(m)` | một dòng, cần chứng minh |

Hash một từ dài `L` tốn `O(L)`, tổng mọi từ là `O(m)`.

## 11. Bộ test

In [12]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<38} PASS ({actual})")

sol = Solution()
check("mẫu 1", sol.wordPattern("abba", "dog cat cat dog"), True)
check("mẫu 2", sol.wordPattern("abba", "dog cat cat fish"), False)
check("mẫu 3", sol.wordPattern("aaaa", "dog cat cat dog"), False)
check("chiều ngược: abba / dog dog dog dog", sol.wordPattern("abba", "dog dog dog dog"), False)
check("pattern dài hơn", sol.wordPattern("aaa", "aa aa"), False)
check("câu dài hơn", sol.wordPattern("aaa", "aa aa aa aa"), False)
check("một chữ một từ", sol.wordPattern("a", "x"), True)
check("chữ trùng tên từ (ab / b a)", sol.wordPattern("ab", "b a"), True)
check("từ là tiền tố nhau", sol.wordPattern("ab", "dog dogs"), True)
check("abc / x x y", sol.wordPattern("abc", "x x y"), False)

mẫu 1                                  PASS (True)
mẫu 2                                  PASS (False)
mẫu 3                                  PASS (False)
chiều ngược: abba / dog dog dog dog    PASS (False)
pattern dài hơn                        PASS (False)
câu dài hơn                            PASS (False)
một chữ một từ                         PASS (True)
chữ trùng tên từ (ab / b a)            PASS (True)
từ là tiền tố nhau                     PASS (True)
abc / x x y                            PASS (False)


In [13]:
import random
rng = random.Random(290)
funcs = [word_pattern_index, word_pattern_two_dicts, word_pattern_dict_set, word_pattern_sets, Solution().wordPattern]

def brute(pattern, s):
    # thử mọi song ánh là không khả thi; dùng định nghĩa "cùng hình dạng" bằng chuẩn hóa
    words = s.split()
    if len(words) != len(pattern):
        return False
    def shape(seq):
        first = {}
        return [first.setdefault(x, len(first)) for x in seq]
    return shape(pattern) == shape(words)

vocab = ["dog", "cat", "fish", "a", "b", "dogs"]
for _ in range(3000):
    n = rng.randint(1, 7)
    p = "".join(rng.choice("abc") for _ in range(n))
    s = " ".join(rng.choice(vocab[:rng.randint(1, 6)]) for _ in range(n + rng.choice([0, 0, 0, -1, 1])) ) or "x"
    exp = brute(p, s)
    assert all(f(p, s) == exp for f in funcs), (p, s)
print("4 cách khớp với chuẩn hóa 'hình dạng' trên 3000 cặp ngẫu nhiên")

4 cách khớp với chuẩn hóa 'hình dạng' trên 3000 cặp ngẫu nhiên


## 12. Lỗi người mới hay gặp

### Lỗi 1: Chỉ kiểm một chiều (chữ → từ)

In [14]:
def wrong_one_way(pattern, s):
    words = s.split()
    if len(pattern) != len(words):
        return False
    c2w = {}
    for c, w in zip(pattern, words):
        if c in c2w and c2w[c] != w:
            return False
        c2w[c] = w
    return True

print("abba / dog dog dog dog ->", wrong_one_way("abba", "dog dog dog dog"), "| đúng = False")

abba / dog dog dog dog -> True | đúng = False


### Lỗi 2: Quên kiểm độ dài (bị `zip` che giấu)

In [15]:
def wrong_no_len(pattern, s):
    c2w, w2c = {}, {}
    for c, w in zip(pattern, s.split()):
        if c2w.get(c, w) != w or w2c.get(w, c) != c:
            return False
        c2w[c] = w; w2c[w] = c
    return True

print("aaa / aa aa aa aa ->", wrong_no_len("aaa", "aa aa aa aa"), "| đúng = False")
print("aaaa / dog ->", wrong_no_len("aaaa", "dog"), "| đúng = False")

aaa / aa aa aa aa -> True | đúng = False
aaaa / dog -> True | đúng = False


### Lỗi 3: So sánh ký tự của `s` thay vì **từ**

Duyệt `zip(pattern, s)` (không `split`) sẽ ghép chữ với **ký tự** của câu, kể cả dấu cách.

In [16]:
print(list(zip("abba", "dog cat cat dog"))[:4], "<- ghép với ký tự, sai hoàn toàn")

[('a', 'd'), ('b', 'o'), ('b', 'g'), ('a', ' ')] <- ghép với ký tự, sai hoàn toàn


### Lỗi 4: Dùng **một** dict chung cho cả chữ và từ

Có người lưu `m[c] = w` và `m[w] = c` vào **cùng** một dict. Khi một từ trùng tên một chữ cái (ví dụ từ `"a"`), key "từ a" đè lên key "chữ a" → đọc nhầm.

In [17]:
def wrong_shared_dict(pattern, s):
    words = s.split()
    if len(pattern) != len(words):
        return False
    m = {}
    for c, w in zip(pattern, words):
        if m.get(c, w) != w or m.get(w, c) != c:
            return False
        m[c] = w
        m[w] = c
    return True

print("ab / c a ->", wrong_shared_dict("ab", "c a"), "| đúng =", Solution().wordPattern("ab", "c a"), "(a->c, b->a là song ánh hợp lệ)")

ab / c a -> False | đúng = True (a->c, b->a là song ánh hợp lệ)


**Sửa:** hai dict tách biệt (hoặc gắn tiền tố cho key, ví dụ `("c", c)` và `("w", w)`).

### Lỗi 5: Dùng `index` trên chuỗi `s` thay vì list từ

`s.index("cat")` trả về vị trí **ký tự**, và `"dog"` có thể là chuỗi con của `"dogs"` → sai lệch. Luôn làm việc trên `words = s.split()`.

In [18]:
s = "dogs dog"
print("s.index('dog') =", s.index("dog"), "(ký tự, khớp nhầm vào 'dogs')", "| words.index('dog') =", s.split().index("dog"))

s.index('dog') = 0 (ký tự, khớp nhầm vào 'dogs') | words.index('dog') = 1


## 13. Mẹo nhận dạng pattern

- **"Cùng mẫu", "isomorphic", "song ánh", "one-to-one mapping"** → hai dict hai chiều, hoặc chuẩn hóa "hình dạng" (thay mỗi phần tử bằng chỉ số lần đầu xuất hiện).
- **So khớp mẫu giữa hai dãy cùng độ dài** → luôn kiểm độ dài trước khi `zip`.
- Cần **nhóm** nhiều chuỗi theo cùng mẫu → dùng "hình dạng" làm key của dict (bài 890).

Bài liên quan: 205 Isomorphic Strings (chữ ↔ chữ), 890 Find and Replace Pattern, 291 Word Pattern II (Hard — không có dấu cách, phải backtracking), 242 Valid Anagram.

## 14. Biến thể

### 14.1. Isomorphic Strings (205) — cùng khuôn, chữ ↔ chữ

In [19]:
def is_isomorphic(s: str, t: str) -> bool:
    if len(s) != len(t):
        return False
    a2b, b2a = {}, {}
    for x, y in zip(s, t):
        if a2b.get(x, y) != y or b2a.get(y, x) != x:
            return False
        a2b[x] = y
        b2a[y] = x
    return True

print(is_isomorphic("egg", "add"), is_isomorphic("foo", "bar"), is_isomorphic("paper", "title"))

True False True


### 14.2. Chuẩn hóa "hình dạng" — dùng làm key để nhóm (Find and Replace Pattern, 890)

In [20]:
def shape(seq):
    first = {}
    return tuple(first.setdefault(x, len(first)) for x in seq)

print(shape("abba"), shape("dog cat cat dog".split()), shape("mee"), shape("aqq"))

def find_and_replace_pattern(words: list[str], pattern: str) -> list[str]:
    target = shape(pattern)
    return [w for w in words if shape(w) == target]

print(find_and_replace_pattern(["abc", "deq", "mee", "aqq", "dkd", "ccc"], "abb"))

(0, 1, 1, 0) (0, 1, 1, 0) (0, 1, 1) (0, 1, 1)
['mee', 'aqq']


`first.setdefault(x, len(first))`: nếu `x` mới thì gán số thứ tự tiếp theo (`len(first)` = số phần tử đã có), nếu cũ thì trả số đã gán. Thứ tự gán = thứ tự xuất hiện lần đầu.

### 14.3. Word Pattern II (291, Hard) — khi **không có dấu cách**

Phải thử mọi cách cắt chuỗi → backtracking. Bản nhỏ để thấy ý tưởng:

In [21]:
def word_pattern_ii(pattern: str, s: str) -> bool:
    c2w, used = {}, set()
    def bt(i, j):
        if i == len(pattern):
            return j == len(s)
        c = pattern[i]
        if c in c2w:
            w = c2w[c]
            return s.startswith(w, j) and bt(i + 1, j + len(w))
        for end in range(j + 1, len(s) + 1):
            w = s[j:end]
            if w in used:
                continue
            c2w[c] = w; used.add(w)
            if bt(i + 1, end):
                return True
            del c2w[c]; used.discard(w)
        return False
    return bt(0, 0)

print(word_pattern_ii("abab", "redblueredblue"), word_pattern_ii("aaaa", "asdasdasdasd"), word_pattern_ii("aabb", "xyzabcxzyabc"))

True True False


## 15. Bài tập (tự làm — cell chỉ định nghĩa, không gọi)

**Bài 1 — Dự đoán:** kết quả của `("abc", "b c a")`, `("aba", "x y y")`, `("abab", "go up go up")`, `("ab", "hi hi")`.

**Bài 2 — Điền chỗ trống:** bản hai dict.

**Bài 3 — Tự viết bản "hình dạng"** (`shape`) cho Word Pattern.

**Bài 4 — Isomorphic Strings** bằng bản một dict + một set.

**Bài 5 — Nhóm câu theo mẫu:** cho list câu, trả về dict `hình dạng → list câu`.

**Bài 6 — Mini challenge:** trả về **ánh xạ** `chữ → từ` nếu có song ánh, ngược lại trả về **chỉ số đầu tiên** gây mâu thuẫn (hoặc `-1` nếu độ dài khác nhau).

In [22]:
# Bài 1 — TODO
predict = None   # list 4 bool
# assert predict == [Solution().wordPattern(p, s) for p, s in [("abc", "b c a"), ("aba", "x y y"), ("abab", "go up go up"), ("ab", "hi hi")]]

In [23]:
def exercise_two_dicts(pattern: str, s: str) -> bool:
    words = s.split()
    # TODO 1: kiểm độ dài
    c2w, w2c = {}, {}
    for c, w in zip(pattern, words):
        # TODO 2: kiểm 2 chiều
        # TODO 3: ghi nhận
        pass
    return True

# assert exercise_two_dicts("abba", "dog dog dog dog") == False and exercise_two_dicts("abba", "dog cat cat dog") == True

In [24]:
def exercise_shape_pattern(pattern: str, s: str) -> bool:
    # TODO
    pass

# assert exercise_shape_pattern("abba", "dog cat cat dog") and not exercise_shape_pattern("aaa", "aa aa aa aa")

In [25]:
def exercise_isomorphic(s: str, t: str) -> bool:
    # TODO
    pass

# assert exercise_isomorphic("egg", "add") and not exercise_isomorphic("badc", "baba")

In [26]:
def exercise_group_by_pattern(sentences: list[str]) -> dict:
    # TODO
    pass

# g = exercise_group_by_pattern(["dog cat cat dog", "a b b a", "x x", "go go"])
# assert sorted(map(sorted, g.values())) == [["a b b a", "dog cat cat dog"], ["go go", "x x"]]

In [27]:
def exercise_mapping_or_conflict(pattern: str, s: str):
    # TODO
    pass

# assert exercise_mapping_or_conflict("abba", "dog cat cat dog") == {"a": "dog", "b": "cat"}
# assert exercise_mapping_or_conflict("abba", "dog cat cat fish") == 3
# assert exercise_mapping_or_conflict("ab", "x") == -1

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | so hình dạng | `abc` → (0,1,2) | `aba` (0,1,0) vs `x y y` (0,1,1) |
| 2 | mục 5 | `c in c2w and c2w[c] != w` | tương tự cho `w2c` |
| 3 | mục 14.2 | so `shape(pattern)` với `shape(words)` | nhớ kiểm độ dài (hoặc để tuple khác độ dài tự khác) |
| 4 | mục 7 | `used` là set ký tự của `t` đã có chủ | `len(s) == len(t)` |
| 5 | key = `shape(câu.split())` | `setdefault(key, []).append(câu)` | |
| 6 | dùng `enumerate` | trả `i` ngay khi mâu thuẫn | hết vòng trả `c2w` |

## 17. Đáp án tham khảo

In [28]:
answer_predict = [True, False, True, False]
assert answer_predict == [Solution().wordPattern(p, s) for p, s in [("abc", "b c a"), ("aba", "x y y"), ("abab", "go up go up"), ("ab", "hi hi")]]
print("Bài 1:", answer_predict)

def answer_two_dicts(pattern, s):
    words = s.split()
    if len(pattern) != len(words):
        return False
    c2w, w2c = {}, {}
    for c, w in zip(pattern, words):
        if (c in c2w and c2w[c] != w) or (w in w2c and w2c[w] != c):
            return False
        c2w[c] = w
        w2c[w] = c
    return True

def answer_shape_pattern(pattern, s):
    return shape(pattern) == shape(s.split())

def answer_isomorphic(s, t):
    if len(s) != len(t):
        return False
    m, used = {}, set()
    for x, y in zip(s, t):
        if x in m:
            if m[x] != y:
                return False
        else:
            if y in used:
                return False
            m[x] = y
            used.add(y)
    return True

def answer_group_by_pattern(sentences):
    groups = {}
    for sen in sentences:
        groups.setdefault(shape(sen.split()), []).append(sen)
    return groups

def answer_mapping_or_conflict(pattern, s):
    words = s.split()
    if len(pattern) != len(words):
        return -1
    c2w, w2c = {}, {}
    for i, (c, w) in enumerate(zip(pattern, words)):
        if c2w.get(c, w) != w or w2c.get(w, c) != c:
            return i
        c2w[c] = w
        w2c[w] = c
    return c2w

assert answer_two_dicts("abba", "dog dog dog dog") == False and answer_two_dicts("abba", "dog cat cat dog") == True
assert answer_shape_pattern("abba", "dog cat cat dog") and not answer_shape_pattern("aaa", "aa aa aa aa")
assert answer_isomorphic("egg", "add") and not answer_isomorphic("badc", "baba")
g = answer_group_by_pattern(["dog cat cat dog", "a b b a", "x x", "go go"])
assert sorted(map(sorted, g.values())) == [["a b b a", "dog cat cat dog"], ["go go", "x x"]]
assert answer_mapping_or_conflict("abba", "dog cat cat dog") == {"a": "dog", "b": "cat"}
assert answer_mapping_or_conflict("abba", "dog cat cat fish") == 3
assert answer_mapping_or_conflict("ab", "x") == -1
for _ in range(1000):
    n = rng.randint(1, 6)
    p = "".join(rng.choice("abc") for _ in range(n))
    s = " ".join(rng.choice(vocab) for _ in range(n))
    assert answer_shape_pattern(p, s) == Solution().wordPattern(p, s) == answer_two_dicts(p, s)
print("Tất cả đáp án PASS")

Bài 1: [True, False, True, False]
Tất cả đáp án PASS


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def wordPattern(self, pattern: str, s: str) -> bool:
        words = s.split()
        if len(pattern) != len(words):
            return False
        c2w, w2c = {}, {}
        for c, w in zip(pattern, words):
            if c2w.get(c, w) != w or w2c.get(w, c) != c:
                return False
            c2w[c] = w
            w2c[w] = c
        return True
```

**Câu thần chú:** *"Song ánh = hai chiều: chữ → từ và từ → chữ; zip thì kiểm độ dài trước."*

**Checklist:**
- [ ] `words = s.split()` — làm việc trên **từ**, không phải ký tự.
- [ ] Kiểm `len(pattern) == len(words)` trước `zip`.
- [ ] Kiểm **cả hai** chiều.
- [ ] Hai dict tách biệt (không dùng chung một dict).
- [ ] Tên method `wordPattern`, trả `bool`.